In [6]:
import numpy as np
import librosa
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
import os

# --- CONFIGURE THESE PATHS ---
MODEL_PATH = 'C:\\Users\\Rhea Dmello\\OneDrive\\Desktop\\Projects -yt\\Bee project\\models\\bee1.h5'
AUDIO_FILE_PATH = "C:\\Users\\Rhea Dmello\\OneDrive\\Desktop\\Projects -yt\\Bee project\\Unseen audio files\\Queenbee present2.mp3"
# -----------------------------

# Check if model exists
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at: {MODEL_PATH}")

if not os.path.exists(AUDIO_FILE_PATH):
    raise FileNotFoundError(f"Audio file not found: {AUDIO_FILE_PATH}")

# Load model
model = load_model(MODEL_PATH)

# Re-create label encoder
encoder = LabelEncoder()
encoder.fit(['bee', 'nobee', 'noqueen'])

# Feature extraction (supports any format via librosa)
def extract_features(file_path, n_mfcc=40, target_sr=16000):
    try:
        # Load audio file — librosa handles .mp3, .wav, .flac, etc.
        audio_data, original_sr = librosa.load(file_path, sr=None)  # Load original SR
        
        # Resample to consistent rate (e.g. 16kHz) — must match training!
        audio_data = librosa.resample(audio_data, orig_sr=original_sr, target_sr=target_sr)
        
        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=audio_data, sr=target_sr, n_mfcc=n_mfcc)
        mfccs_scaled = np.mean(mfccs.T, axis=0)  # Average over time
        return mfccs_scaled
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# --- PREDICTION ---
print(f"🎧 Loading audio: {os.path.basename(AUDIO_FILE_PATH)}")
features = extract_features(AUDIO_FILE_PATH)

if features is not None:
    # Reshape for model input (1 sample, n_mfcc features)
    features = features.reshape(1, -1)
    
    # Predict
    prediction = model.predict(features, verbose=0)
    predicted_class = encoder.inverse_transform([np.argmax(prediction)])[0]
    confidence = np.max(prediction) * 100

    print(f"\n✅ Predicted Class: {predicted_class.upper()}")
    print(f"📊 Confidence: {confidence:.2f}%")
else:
    print("❌ Failed to extract features from audio.")

🎧 Loading audio: Queenbee present2.mp3



✅ Predicted Class: BEE
📊 Confidence: 99.16%
